# Comparativo de estrategias de chunking

**LAB-26 · F1-09** · Este notebook responde a una pregunta concreta de nuestro RAG: *¿cambia lo que recupera el
asistente según cómo se corten los documentos en fragmentos (chunks)?*

Se comparan **tres estrategias** sobre el mismo corpus (`data/kb`, 24 políticas y FAQs de ShopLab en español):

| Estrategia | Cómo corta |
|---|---|
| **Tamaño fijo** | cada `chunk_size` caracteres, con solape; no mira el contenido |
| **Por encabezados** | por secciones del Markdown; es el troceado de producción de `ingest/` (ADR-0016) |
| **Semántica** | donde cambia el significado entre frases consecutivas, medido con embeddings |

Después se lanzan **20 preguntas de prueba** contra los chunks de cada estrategia y se mide si el chunk correcto
aparece entre los `top_k` recuperados.

**Cómo leerlo:** las celdas se ejecutan de arriba abajo. La celda 1 declara todos los parámetros; la 2 comprueba que
Ollama y el modelo de embeddings están disponibles (si no, se detiene antes de calcular nada). Este notebook **mide y
concluye**, pero no cambia el chunking de producción.

In [1]:
from chunking_lab.config import ExperimentConfig

config = ExperimentConfig(
    chunk_size=1000,           # tamaño máximo de un chunk en caracteres; igual en las tres estrategias
    chunk_overlap=100,         # caracteres repetidos entre chunks consecutivos (tamaño fijo y encabezados)
    top_k=4,                   # chunks recuperados por pregunta; el chat de producción usa 4
    breakpoint_percentile=80,  # semántica: corta donde la distancia supera este percentil del documento
    min_chunk_chars=200,       # semántica: los fragmentos más cortos se unen al vecino más cercano
)
# El modelo de embeddings y la URL salen de OLLAMA_EMBED_MODEL y OLLAMA_BASE_URL (bge-m3 y localhost:11434).
print(f"modelo={config.embed_model} · chunk_size={config.chunk_size} · chunk_overlap={config.chunk_overlap}"
      f" · top_k={config.top_k} · percentil={config.breakpoint_percentile} · min_chunk_chars={config.min_chunk_chars}")

modelo=bge-m3 · chunk_size=1000 · chunk_overlap=100 · top_k=4 · percentil=80 · min_chunk_chars=200


In [2]:
from chunking_lab.embeddings import OllamaEmbedder

embedder = OllamaEmbedder(config)
model_digest = embedder.preflight()  # PreflightError, con instrucciones, si Ollama o el modelo no están
print(f"Ollama disponible · modelo {config.embed_model} · digest {model_digest[:12] or 'no informado'}")

Ollama disponible · modelo bge-m3 · digest 790764642607


## 1. El corpus

Se usan los documentos Markdown de la base de conocimiento, sin modificarlos. Los PDF se excluyen porque repiten el
texto de su Markdown y aquí no se compara el parseo de PDF.

In [3]:
import pandas as pd

from chunking_lab.corpus import load_corpus

documents = load_corpus(config.kb_dir)
corpus_table = (
    pd.DataFrame({"tema": [d.topic for d in documents], "caracteres": [len(d.body) for d in documents]})
    .groupby("tema").agg(documentos=("tema", "size"), caracteres=("caracteres", "sum"))
)
corpus_table.loc["total"] = corpus_table.sum()
print(f"{len(documents)} documentos; media de {sum(len(d.body) for d in documents) // len(documents)} caracteres")
corpus_table

24 documentos; media de 1442 caracteres


,documentos,caracteres
tema,,
payments,5,6923
privacy,5,7056
returns,5,7839
shipping,5,7084
warranty,4,5727
total,24,34629


## 2. Las tres estrategias

Las tres se aplican con el mismo `chunk_size`. Cada una devuelve chunks con su documento y su posición.

* **Tamaño fijo** corta en el carácter número `chunk_size`, aunque sea a mitad de palabra, de frase o de tabla.
* **Por encabezados** nunca mezcla dos secciones y conserva la ruta de encabezados de cada chunk; si una sección es
  más larga que `chunk_size`, la divide en párrafos, líneas y frases.
* **Semántica** parte el texto en unidades (frases, elementos de lista, tablas), calcula con `bge-m3` la distancia entre
  unidades consecutivas y corta donde esa distancia supera el percentil `breakpoint_percentile` del propio documento.
  Si un grupo excede `chunk_size` lo vuelve a partir en su mayor distancia interna, y si es menor que `min_chunk_chars`
  lo une al vecino más cercano. No usa solape.

In [4]:
from chunking_lab.strategies import Strategy, build_chunks

chunks = {strategy: build_chunks(strategy, documents, config, embedder) for strategy in Strategy}
{strategy.value: len(found) for strategy, found in chunks.items()}

{'fixed': 48, 'headings': 138, 'semantic': 59}

### Cómo decide cortar la estrategia semántica

La curva muestra, para un documento, la distancia entre cada unidad y la siguiente. Los puntos rojos por encima del
umbral son los cortes: ahí el significado cambia más que en el resto del documento.

In [5]:
from chunking_lab.report import plot_distance_curve
from chunking_lab.strategies.semantic import adjacent_distances, breakpoint_threshold, semantic_units

example_document = next(d for d in documents if d.id == "returns-01-return-policy")
units = semantic_units(example_document.body, config.chunk_size)
distances = adjacent_distances(units, embedder)
threshold = breakpoint_threshold(distances, config.breakpoint_percentile)
plot_distance_curve(distances, threshold, f"{example_document.id}: {len(units)} unidades")

<Figure size 800x280 with 1 Axes>

## 3. Cuántos chunks produce cada estrategia

Con documentos de pocos miles de caracteres, el tamaño máximo por defecto (1 000) hace que el **tamaño fijo** produzca
pocos chunks casi del tamaño de un documento, mientras que **encabezados** produce muchos chunks pequeños. Esta
diferencia de granularidad forma parte de lo que se compara.

In [6]:
import pandas as pd

from chunking_lab.report import STRATEGY_LABELS, plot_size_histograms, size_summary

sizes = size_summary(chunks)
size_table = pd.DataFrame(sizes).T
size_table.index = [STRATEGY_LABELS[s] for s in size_table.index]
size_table.columns = ["chunks", "mínimo", "media", "máximo"]
size_table = size_table.astype({"chunks": int, "mínimo": int, "máximo": int})
display(size_table)
plot_size_histograms(chunks, config.chunk_size)

,chunks,mínimo,media,máximo
Tamaño fijo,48,225,771.4,1000
Por encabezados,138,72,242.3,731
Semántica,59,191,582.4,997


<Figure size 1200x300 with 3 Axes>

## 4. Un mismo pasaje, tres cortes

Un pasaje cuya respuesta cae cerca de un límite de chunk. Abajo se ve, para cada estrategia, qué chunk(s) contienen su
inicio y si el pasaje completo quedó **partido** (ningún chunk lo contiene entero).

In [7]:
from chunking_lab.report import cut_example
from chunking_lab.text import normalize

example_id = "payments-02-installments"
example_passage = (
    "El número de cuotas disponible depende del total del pedido, porque cada cuota debe ser de al menos R$ 30,00."
)
for strategy, example in cut_example(chunks, example_id, example_passage).items():
    print(f"=== {STRATEGY_LABELS[strategy]} · {'PASAJE PARTIDO' if example.cut else 'pasaje entero en un chunk'} ===")
    for found in example.chunks:
        text = normalize(found.text)
        start = text.find(normalize(example_passage)[:20])
        print(f"[chunk {found.position}, {len(found.text)} caracteres]")
        print("  ..." + text[max(0, start - 60):start + len(example_passage) + 40] + "...")
    print()

=== Tamaño fijo · PASAJE PARTIDO ===
[chunk 0, 1000 caracteres]
  ...nterés; la de 4 cuotas tiene interés. ## Casos especiales - El número de cuotas disponible depende del total del pedido, porque cada cuota debe ser de al menos R$ 30,0...

=== Por encabezados · pasaje entero en un chunk ===
[chunk 3, 297 caracteres]
  ...Casos especiales - El número de cuotas disponible depende del total del pedido, porque cada cuota debe ser de al menos R$ 30,00. - Si devuelves el pedido, el reembolso ...

=== Semántica · pasaje entero en un chunk ===
[chunk 0, 995 caracteres]
  ...nterés; la de 4 cuotas tiene interés. ## Casos especiales - El número de cuotas disponible depende del total del pedido, porque cada cuota debe ser de al menos R$ 30,00....



## 5. Las 20 preguntas de prueba

Son veinte preguntas en español, **cuatro por cada tema**, escritas como las haría un cliente. Cada una lleva el
documento donde está la respuesta y el **pasaje de referencia**, el texto literal que la contiene. Se guardan como datos
(`notebooks/data/questions.toml`), separados de este código.

Antes de medir nada se validan: 20 preguntas, 4 por tema, ids únicos, que el pasaje exista en su documento y **solo** en
ese, y que mida entre 40 y 300 caracteres. Si alguna regla falla, la celda se detiene y las lista todas.

In [8]:
from chunking_lab.questions import load_questions, validate_questions

questions = load_questions(config.questions_path)
validate_questions(questions, documents)
pd.DataFrame([{"id": q.id, "tema": q.topic, "pregunta": q.question, "documento esperado": q.document}
              for q in questions]).set_index("id")

,tema,pregunta,documento esperado
id,,,
Q01,payments,"Si mi compra es de R$ 120, ¿puedo pagarla en 5...",payments-02-installments
Q02,payments,"Mi boleto venció sin que lo pagara, ¿qué puedo...",payments-03-boleto-and-vouchers
Q03,payments,Ya intenté pagar tres veces el mismo pedido y ...,payments-04-payment-issues
Q04,payments,"Veo un cobro de ShopLab que no reconozco, ¿qué...",payments-04-payment-issues
Q05,privacy,¿Cómo borro las cookies que el sitio ya guardó...,privacy-04-cookies
Q06,privacy,¿Puedo darle mi contraseña al asistente para q...,privacy-02-data-collection-and-use
Q07,privacy,¿Qué información debo incluir en el correo par...,privacy-03-data-rights
Q08,privacy,¿Con quién comparte ShopLab mis datos personales?,privacy-01-privacy-policy
Q09,returns,¿Qué tengo que indicar al pedir la devolución ...,returns-02-return-process


## 6. Recuperación y tabla de resultados

Para cada estrategia se calcula el embedding de todos sus chunks y de cada pregunta (el mismo modelo en las tres), y se
recuperan los `top_k` chunks más cercanos por coseno **entre todos los documentos del corpus**, como hace el chat.

Un chunk recuperado cuenta como **correcto solo si procede del documento esperado y contiene el pasaje de referencia**;
un chunk del documento correcto que no incluye el pasaje no cuenta. Para cada pregunta se anota la **posición** (de 1 a
`top_k`) del primer chunk correcto, o «no encontrado». Con eso:

* **hit rate@k**: fracción de preguntas con algún chunk correcto entre los `top_k`.
* **MRR**: media de `1 / posición` (0 si no se encontró); premia que el correcto salga primero.

In [9]:
from chunking_lab.retrieval import ChunkIndex, run_retrieval

retrievals = {
    strategy: run_retrieval(ChunkIndex.build(chunks[strategy], embedder), questions, embedder, config.top_k)
    for strategy in Strategy
}
ranks = {strategy: {r.question_id: r.rank for r in found} for strategy, found in retrievals.items()}
{STRATEGY_LABELS[s]: sum(1 for r in found.values() if r is not None) for s, found in ranks.items()}

{'Tamaño fijo': 18, 'Por encabezados': 17, 'Semántica': 17}

In [10]:
from datetime import UTC, datetime

from chunking_lab.report import build_results, plot_metrics, results_table, validate_results

results = build_results(config, sizes, ranks, questions, model_digest, len(documents),
                        datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"))
validate_results(results)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(results_table(results).set_index("pregunta"))
plot_metrics(results)

,tema,documento esperado,Tamaño fijo,Por encabezados,Semántica
pregunta,,,,,
Q01,payments,payments-02-installments,1,2,1
Q02,payments,payments-03-boleto-and-vouchers,1,3,1
Q03,payments,payments-04-payment-issues,no encontrado,no encontrado,no encontrado
Q04,payments,payments-04-payment-issues,1,1,1
Q05,privacy,privacy-04-cookies,1,1,2
Q06,privacy,privacy-02-data-collection-and-use,1,1,1
Q07,privacy,privacy-03-data-rights,1,2,1
Q08,privacy,privacy-01-privacy-policy,1,2,2
Q09,returns,returns-02-return-process,1,4,1


<Figure size 600x300 with 1 Axes>

## Resultados guardados

Los resultados se escriben en `notebooks/results/chunking-results.json` y se imprime su huella `sha256`. Una prueba
automática comprueba que este notebook y ese archivo siguen siendo coherentes.

In [11]:
from chunking_lab.report import results_digest, write_results

write_results(results, config.results_path)
print(f"Resultados guardados en notebooks/results/{config.results_path.name}")
print(f"sha256: {results_digest(config.results_path)}")

Resultados guardados en notebooks/results/chunking-results.json
sha256: 11a0a5f1cf5942c7fae9efef5d04c0664eae20ba3e14c250d5d29c434e911be9
